In this notebook, we walk through how to resynthesize a multi-frame TSV using `plot_waves`. Let's load a TSV first:

In [1]:
import pandas as pd

# Load data from TSV

new_df = pd.read_csv('../tsv/conga_multidft.tsv', delimiter='\t')

display(new_df)

,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,71.428571,71.428571,0.03,0.10,0.481900,0.481900
1,128.571429,128.571429,0.03,0.10,0.525280,0.525280
2,157.142857,157.142857,0.03,0.10,0.556550,0.556550
3,200.000000,200.000000,0.03,0.10,0.567891,0.567891
4,242.857143,242.857143,0.03,0.10,0.740521,0.740521
...,...,...,...,...,...,...
82,395.000000,395.000000,0.20,0.40,0.051115,0.051115
83,515.000000,515.000000,0.20,0.40,0.021439,0.021439
84,550.000000,550.000000,0.20,0.40,0.018971,0.018971
85,600.000000,600.000000,0.20,0.40,0.043340,0.043340


In [2]:
from audiospylt.multiplotter import plot_combined_3d

plot_combined_3d(dfs=[new_df],
    axis_order=("time", "amp", "freq"),
    flip={"time": True},
)  # if df also has amp_min/amp_max columns

Based on the highest frequency in our TSV example, we can estimate an appropriate sampling rate to use for rendering:

In [3]:
from audiospylt.plot_wave import estimate_sampling_frequency_and_time_vector

sampling_frequency_updated, delta_t_updated, duration = estimate_sampling_frequency_and_time_vector(new_df)

print(f"Estimated Optimal Sampling Frequency: {sampling_frequency_updated:.3f} Hz")
print(f"Sampling Interval (Delta t): {delta_t_updated:.6f} seconds")
print(f"Duration (highest value from time_stop column): {duration:.3f} seconds")


Estimated Optimal Sampling Frequency: 6342.857 Hz
Sampling Interval (Delta t): 0.000158 seconds
Duration (highest value from time_stop column): 0.750 seconds


We can treat the estimated sampling frequency as guidance, but we can freely change the time step (`k`) to experiment:

In [4]:
from audiospylt.plot_wave import plot_waves

# Define the time step k (seconds) and corresponding sample rate
k=0.0005
sample_rate = 1.0 / k

# Optional FFT/spectrogram settings (forwarded to `audiospylt.audio_utils.plot_spectrogram`)
spectrogram_kwargs = {
    # Y-axis scaling
    "y_axis_mode": "log",           # 'linear' | 'log' | 'mel' | 'mixed'
    "y_axis_mix": 0.5,               # 0..1 (only for 'mixed': blends linear<->log spacing)
    "mixed_log_floor_hz": 1.0,       # >0 (only for 'mixed': log floor; keep >0 to avoid log(0))

    # FFT / time-frequency settings
    "n_fft": 256,                    # window/FFT size; larger -> finer frequency bins, coarser time bins
    "window_type": "hann",          # window shape (scipy.signal.get_window)
    "overlap": 0.75,                 # fraction of window overlap; keep <0.95 (hop must stay >0)
    "oversample_factor": 1.0,        # >=1.0; zero-pads the FFT (more interpolated frequency grid, same time grid)

    # Mel settings (only for y_axis_mode='mel')
    "mel_bins": 128,
    "mel_fmax": sample_rate / 2,     # cap at Nyquist

    # Rendering / framing (affects both computation and how the plot aligns in time)
    "scaling": "density",           # SciPy scaling: 'density' (PSD-like, per Hz) | 'spectrum' (per-bin)
    "mode": "magnitude",            # 'magnitude' = |STFT|, 'psd' = power spectral density (scaling applies)
    "cmap": "Magma",                # Plotly continuous colorscale name (e.g. 'Viridis', 'Magma', ...)

    # STFT edge handling:
    # - None: no padding ("valid" framing); first/last ~n_fft/2 may be missing in the visible extent
    # - 'zeros': pad both sides by n_fft//2 so a frame can be centered at t=0 (shows pre-zero padding)
    # - 'zeros_end': pad only the end so the first full window starts at t=0 (often cleaner for onsets)
    # - any other string: forwarded as `numpy.pad` mode (pads both sides by n_fft//2)
    "boundary": "zeros",

    # If True, add minimal end-padding so the last hop boundary inside the *original* signal
    # still gets a frame (helps avoid an overlap-dependent "gap" near the end).
    "padded": True,

    # X-axis view range:
    # - 'signal': force x-axis to [0, duration] (aligns with waveform)
    # - 'stft': show the full extent of generated frames (can include padding tails)
    "time_range": "signal",

    "show": True,                    # show the Plotly figure now (`plot_waves` defaults to `True` if omitted)
}

# Default behavior: show the synthesized waveform only (time-step `k` resolution intact).
# Set show_fft=True when you want the spectrogram.
y_combined = plot_waves(
    new_df,
    k,
    # Convenience fade duration (seconds) applied to *both* ends of each event.
    # - 0 disables per-event edge fades (useful when you want exact hard boundaries)
    # - You can override per-row with df columns 'fade_in_s' / 'fade_out_s'
    # - Or pass fade_in_s=... / fade_out_s=... to control globally (takes precedence over edge_fade_s)
    edge_fade_s=0.0025,
    # Phase handling inside each event:
    # - 'reset': phase restarts at each event start (simple; most prone to boundary clicks)
    # - 'global': absolute-time-referenced phase; keeps phase continuous across adjacent events
    #             when freq_start==freq_stop (constant-frequency bins)
    # - 'chirp': integrates a linear ramp so instantaneous frequency follows
    #             freq_start->freq_stop *within* the event (does not guarantee continuity across events)
    phase_mode="chirp",
    show_waveform=True,
    show_fft=True,
    spectrogram_kwargs=spectrogram_kwargs,
)

We can also intentionally render above the Nyquist limit with `alias_above_nyquist`, which uses `np.interp` (no anti-aliasing low-pass filter). When downsampling, this produces the classic aliasing "mirror around Nyquist" effect.

In [ ]:
from audiospylt.generate_wave_file import render_audio

player = True
save_audio = False

fs_initial = 1 / k  # The initial sampling rate is the inverse of the time step (k)

render_audio(
    y_combined,
    fs_initial,
    fs_target_name="44.1kHz",

    # If True, downsampling to fs_target_name will *not* low-pass above Nyquist.
    # Instead it intentionally allows aliasing ("Nyquist mirroring") so partials above
    # fs_target/2 fold back into-band while the exported file stays at a common sample rate.
    alias_above_nyquist=True,

    bit_rate=24,
    filename_template="testing_{fs_target_name}_{bit_rate}bit_{timestamp}",
    timestamp_format="%Y-%m-%d_%H-%M-%S",
    save_audio=save_audio,
    player=player,
    sanitize=True,
    verbose=True,
);
